# 📊 Demo 04 — Évaluation RAGAS FinRAG

Évaluation complète du système RAG avec le framework RAGAS.

**Métriques :**
- **Faithfulness** : Fidélité aux sources
- **Answer Relevancy** : Pertinence des réponses
- **Context Recall** : Couverture du contexte
- **Context Precision** : Précision du contexte

In [ ]:
import sys
import json
from pathlib import Path

ROOT_DIR = Path('.').resolve().parent
sys.path.insert(0, str(ROOT_DIR))

from src.retrieval.vector_store import FinancialVectorStore
from src.retrieval.retriever import HybridFinancialRetriever
from src.retrieval.reranker import CrossEncoderReRanker
from src.generation.generator import FinancialAnswerGenerator
from src.agents.rag_agent import FinancialRAGAgent
from src.evaluation.evaluator import RAGASEvaluator

vs = FinancialVectorStore()
retriever = HybridFinancialRetriever(vector_store=vs)
reranker = CrossEncoderReRanker()
agent = FinancialRAGAgent(vs, retriever, reranker)

print('Système initialisé')

## 1. Chargement du Dataset d'Évaluation

In [ ]:
import pandas as pd

eval_path = ROOT_DIR / 'data' / 'samples' / 'eval_questions.json'
with open(eval_path) as f:
    eval_data = json.load(f)

df_eval = pd.DataFrame(eval_data)
print(f'Dataset: {len(df_eval)} questions')
print()
print(df_eval[['id', 'question', 'category']].to_string(index=False))

## 2. Évaluation Batch

In [ ]:
evaluator = RAGASEvaluator(
    agent=agent,
    eval_dataset_path=str(eval_path)
)

print('Démarrage évaluation (5 questions)...')
report = evaluator.evaluate_batch(max_samples=5, save_report=True)

print(f'\n=== RAPPORT D\'ÉVALUATION ===')
print(f'Échantillons: {report.n_samples}')
print(f'Faithfulness:      {report.avg_faithfulness:.3f}')
print(f'Answer Relevancy:  {report.avg_answer_relevancy:.3f}')
print(f'Context Recall:    {report.avg_context_recall:.3f}')
print(f'Context Precision: {report.avg_context_precision:.3f}')
print(f'Score Global:      {report.overall_score:.3f}')

## 3. Visualisation des Métriques

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

metrics = {
    'Faithfulness': report.avg_faithfulness,
    'Answer Relevancy': report.avg_answer_relevancy,
    'Context Recall': report.avg_context_recall,
    'Context Precision': report.avg_context_precision,
}

fig = make_subplots(
    rows=1, cols=4,
    specs=[[{'type': 'indicator'}] * 4],
    subplot_titles=list(metrics.keys())
)

colors = ['#3B82F6', '#F0B429', '#10B981', '#EF4444']
for i, (name, value) in enumerate(metrics.items(), 1):
    fig.add_trace(
        go.Indicator(
            mode='gauge+number',
            value=value * 100,
            gauge={
                'axis': {'range': [0, 100]},
                'bar': {'color': colors[i-1]},
                'steps': [
                    {'range': [0, 50], 'color': '#1a0a0a'},
                    {'range': [50, 75], 'color': '#1a1a0a'},
                    {'range': [75, 100], 'color': '#0a1a0a'},
                ],
                'threshold': {'value': 75, 'line': {'color': 'white', 'width': 2}},
            },
            number={'suffix': '%', 'font': {'size': 20}},
        ),
        row=1, col=i
    )

fig.update_layout(
    height=300,
    paper_bgcolor='#0A0E1A',
    font_color='white',
    title_text='Métriques RAGAS — FinRAG',
)
fig.show()

## 4. Analyse par Question

In [ ]:
if report.samples:
    records = []
    for s in report.samples:
        avg = (s.faithfulness + s.answer_relevancy + s.context_recall + s.context_precision) / 4
        records.append({
            'question': s.question[:60] + '...',
            'faithfulness': round(s.faithfulness, 3),
            'answer_relevancy': round(s.answer_relevancy, 3),
            'context_recall': round(s.context_recall, 3),
            'context_precision': round(s.context_precision, 3),
            'overall': round(avg, 3),
        })
    df_results = pd.DataFrame(records)
    print(df_results.to_string(index=False))
else:
    print('Aucun échantillon évalué.')

## 5. Rapport Markdown

In [ ]:
from IPython.display import Markdown, display
display(Markdown(report.to_markdown()))